# 29 — Düşük kontrastta kimlik kaymasını azaltan termal EdgeTAM

Bu notebook tercihen `32_aerial_thermal_stage_b_stable.ipynb`, o yoksa
korunan `22_thermal_deep_3_fixed.ipynb` çıktısını **başlangıç checkpoint'i**
olarak alır ve onu kısa termal video klipleri üzerinde devam eğitir.
Stage B tek kare maskesini öğrenir; ancak
bellek bankasını hiç çalıştırmadığı için, düşük kontrastlı bir kareden
sonra benzer görünen başka bir nesneye kaymayı ne eğitebilir ne de ölçebilir.

Buradaki düzeltme üç parçalıdır:

1. Gerçek veride hedef–yerel arka plan kontrastı düşük diziler daha sık
   örneklenir.
2. Eğitim kliplerinde kontrast ve parlaklık **zamanda yumuşak** değiştirilir;
   hedef bölgesi ayrıca çevresindeki arka plan tonuna yaklaştırılır. Maske ve
   kutu geometrisi değişmez.
3. Model kendi tahminini sonraki karelerin belleğine yazar; son değerlendirme
   aynı checkpoint'i stok bellek ve SAMURAI hareket/bellek kapısıyla ayrı ayrı
   ölçer.

Araştırma dayanağı:

- [SAM 2](https://arxiv.org/abs/2408.00714) video tahminini akış belleğiyle
  koşullandırır; bu yüzden tek kare skoru takip kanıtı değildir.
- [SAMURAI](https://arxiv.org/abs/2411.11922) benzer görünümlü nesnelerde
  görünüş benzerliğinin uzamsal-zamansal tutarlılığı yenebildiğini ve kötü
  karelerin belleğe seçimsiz yazılmasının hatayı yaydığını gösterir.
- [DAM4SAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Videnovic_A_Distractor-Aware_Memory_for_Visual_Object_Tracking_with_SAM2_CVPR_2025_paper.html)
  SAM2 için dikkat dağıtıcı-farkındalıklı bellek ve güvenilir güncellemenin
  gerekli olduğunu doğrudan ölçer.
- [NLMTrack](https://arxiv.org/abs/2407.08265) termal görüntülerde düşük
  kontrast ve az dokunun, zamansal/koordinat bilgisini özellikle önemli
  yaptığını raporlar.

**Girdi:** Drive'daki
`edgetam-stage-b/aerial_thermal_stable/`
`edgetam_pool_aerial_thermal_stable_512.pt`.

**Çıktı:** `edgetam_thermal_contrast_tracking_512.pt`, val/test karşılaştırma
JSON'ları ve SAMURAI'li dağıtım YAML'ı. Orijinal `22` notebook'u ve onun
checkpoint'i değiştirilmez.

In [ ]:
# --- Runtime, repo, GPU -------------------------------------------------
import os, sys
from pathlib import Path

REPO   = Path("/content/sam-dedection")
BRANCH = "claude/thermal-stage-b-training-43ktcl"

if not REPO.exists():
    !git clone -q -b {BRANCH} https://github.com/yigitkayabagci/sam-dedection.git {REPO}
!git -C {REPO} fetch -q origin {BRANCH}
!git -C {REPO} checkout -q {BRANCH} && git -C {REPO} merge -q --ff-only origin/{BRANCH}
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /content | tail -1

In [ ]:
# --- Dependencies -------------------------------------------------------
# EdgeTAM (as `sam2`) and transformers' own SAM2 coexist deliberately: the
# teacher goes through transformers, so no second runtime is needed.
!bash scripts/setup_edgetam.sh 2>&1 | tail -5
!pip install -q -r requirements.txt
!pip install -q "transformers>=4.56" gdown

import sam2, transformers
assert hasattr(transformers, "Sam2Model"), "transformers is too old for the SAM 2.1 teacher"
print(f"sam2 (EdgeTAM) {Path(sam2.__file__).parent}\ntransformers {transformers.__version__}")
assert Path("third_party/EdgeTAM/checkpoints/edgetam.pt").is_file(), \
    "edgetam.pt did not download -- rerun scripts/setup_edgetam.sh and read its output"

In [ ]:
# The contracts everything below depends on, tested with no GPU and no
# checkpoint. If these fail, nothing after this point is worth running.
!python -m unittest tests.test_clip_loop tests.test_training_losses \
    tests.test_antiuav_dataset tests.test_accuracy tests.test_pseudo_labels \
    tests.test_loader tests.test_fetch_antiuav410 2>&1 | tail -3

## Ayarlar

`BASE_STAGE_B`, tercihen notebook 32'nin ürettiği checkpoint olmalıdır. Bu dosya
bulunamazsa stok EdgeTAM'e sessizce düşülmez; aksi halde deney, kullanıcının
iyi şekil tanıyan modelini düzeltmek yerine farklı bir modeli eğitmiş olur.

Kontrast dönüşümü rastgele kare-kare titreşmez. Başlangıç ve bitiş değerleri
çekilir, klip boyunca doğrusal geçiş uygulanır. Böylece termal kameranın AGC
değişimini taklit ederken sahte temporal flicker öğretilmez.

In [ ]:
DATA_DIR = Path("/content/data")
WORK     = Path("/content/work/thermal_contrast_tracking")
SPLITS   = ("train", "val", "test")

SIZE                   = 512
CLIP_LEN, CLIP_STRIDE  = 8, 2
TRAIN_SEQUENCES        = 80
VAL_SEQUENCES          = 20
TEST_SEQUENCES         = 16

TEACHER_ID   = "facebook/sam2.1-hiera-large"
LABEL_STRIDE = 3
ZOOM, MIN_CROP = 4.0, 128

STEPS_PER_EPOCH = 500
VAL_BATCHES     = 32
BATCH_CEILING   = 128
LOADER_WORKERS  = min(2 * (os.cpu_count() or 4), 24)
PREFETCH_DEPTH  = 2
SEED            = 0

# Gerçek düşük-kontrast dizi örneklemesi.
CONTRAST_AUDIT_STRIDE = 40
LOW_CONTRAST_QUANTILE = 0.40
LOW_CONTRAST_REPEAT   = 2

# Zamansal termal bozulma. 0.25 olasılıkla klip tamamen temiz kalır.
AUGMENT_PROB          = 0.75
GLOBAL_CONTRAST       = (0.35, 0.90)
TARGET_CONTRAST       = (0.15, 0.65)
BRIGHTNESS_SHIFT      = (-0.05, 0.05)
SENSOR_NOISE          = (0.0, 0.018)
BLUR_PROB             = 0.20

# SAMURAI bellek kapısı; önce val'de stok bellekle A/B ölçülür.
SAMURAI = {
    "enabled": True, "kf_weight": 0.15,
    "stable_frames": 15, "stable_iou": 0.3,
    "memory_iou": 0.5, "memory_obj_score": 0.0,
    "memory_kf_score": 0.0,
}

LABELS = WORK / "labels"
CKPT   = REPO / "checkpoints"
CHECKPOINT = CKPT / "edgetam_thermal_contrast_tracking_512.pt"
for directory in (DATA_DIR, WORK, LABELS, CKPT):
    directory.mkdir(parents=True, exist_ok=True)

import torch
VRAM = (torch.cuda.get_device_properties(0).total_memory / 2**30
        if torch.cuda.is_available() else 0)
TEACHER_BATCH = 8 if VRAM < 24 else 16 if VRAM < 48 else 32

MIRROR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MIRROR = Path("/content/drive/MyDrive/edgetam-stage-c/thermal_contrast_tracking")
    MIRROR.mkdir(parents=True, exist_ok=True)
except Exception as exc:
    print(f"Drive bağlanamadı ({type(exc).__name__}): {exc}")

BASE_STAGE_B_CANDIDATES = [
    Path("/content/drive/MyDrive/edgetam-stage-b/aerial_thermal_stable/"
         "edgetam_pool_aerial_thermal_stable_512.pt"),
    Path("/content/drive/MyDrive/edgetam-stage-b/thermal_deep/"
         "edgetam_pool_thermal_deep_512.pt"),
]
BASE_STAGE_B = next(
    (path for path in BASE_STAGE_B_CANDIDATES if path.is_file()),
    BASE_STAGE_B_CANDIDATES[0])
assert BASE_STAGE_B.is_file(), (
    f"Stage-B checkpoint bulunamadı: {BASE_STAGE_B_CANDIDATES}. "
    "Önce notebook 32'yi çalıştırın veya yolu doğru .pt dosyasına yöneltin.")

print(f"{VRAM:.0f} GiB VRAM | teacher batch {TEACHER_BATCH}")
print(f"base stage B: {BASE_STAGE_B}")
print(f"output: {CHECKPOINT}")

## Getting the data

8.7 GB from the authors' Google Drive, unpacked in place. `tools/fetch_antiuav410.py`
does three things a `gdown | unzip` does not:

- **checks what it downloaded** — Drive answers a quota-exceeded request with an
  HTML page and an HTTP 200, which becomes a 3 KB "zip" that fails twenty
  minutes later with an unrelated error;
- **extracts in parallel** — 440K members, and zlib releases the GIL;
- **verifies the layout it produced**, rather than assuming the archive's
  top-level folder is spelled the way the README spells it.

It is safe to re-run: an already-extracted split is skipped. If Drive's daily
quota for the file is spent, add it to your own Drive from the [share
link](https://drive.google.com/file/d/1zsdazmKS3mHaEZWS2BnqbYHPEcIaH5WR/view),
mount it, and pass `--zip /content/drive/MyDrive/Anti-UAV410.zip`.

In [ ]:
!python tools/fetch_antiuav410.py --dest {DATA_DIR} --splits {" ".join(SPLITS)}

In [ ]:
from tools.fetch_antiuav410 import dataset_root, describe, find_splits

splits = find_splits(DATA_DIR)
missing = [s for s in SPLITS if s not in splits]
assert not missing, f"{missing} did not extract -- rerun the cell above"

DATA = dataset_root(splits)      # what --data and list_sequences() both want
print(describe(splits))
print(f"\nDATA = {DATA}")

In [ ]:
# --- What is actually in it --------------------------------------------
import numpy as np
from src.training import frame_shape, list_sequences

train = list_sequences(DATA, "train")[:TRAIN_SEQUENCES]
val   = list_sequences(DATA, "val")[:VAL_SEQUENCES]
height, width = frame_shape(train[0].frames[0])

for name, sequences in (("train", train), ("val", val)):
    frames  = sum(len(s) for s in sequences)
    visible = sum(int(s.labels.exist.sum()) for s in sequences)
    print(f"{name:<6} {len(sequences):>3} sequences  {frames:>7} frames  "
          f"{visible:>7} annotated ({visible / max(frames, 1):.1%})")

print(f"\nframe size {width}x{height}  ->  a {SIZE} crop is "
      f"{'native pixels, no resize' if min(width, height) >= SIZE else 'UPSCALED'}")
assert min(width, height) >= SIZE, \
    f"a {SIZE} window does not fit in a {width}x{height} frame; lower SIZE"

In [ ]:
# --- How big are the targets, and do clips stay on native pixels? -------
# The first decides whether notebook 05 (size-adaptive inference) is worth its
# latency. The second is a property of this dataset the training loop has to
# live with: a clip gets a fixed 512 window if the target's whole excursion
# fits in one, and otherwise falls back to resizing the entire frame. The
# window is fixed for the clip on purpose -- SAM 2's memory bank stores
# features in input coordinates, so a window that moved between frames would
# put every stored memory in a different frame of reference from its reader.
from src.training import sample_clips

sides = np.concatenate([
    np.nanmax(np.stack([s.labels.boxes[:, 2] - s.labels.boxes[:, 0],
                        s.labels.boxes[:, 3] - s.labels.boxes[:, 1]]), axis=0)
    for s in train
])
sides = sides[np.isfinite(sides)]
print(f"{len(sides)} annotated targets, longer side in source pixels:")
for lo, hi, name in [(0, 8, "tiny"), (8, 16, "small"), (16, 32, "medium"), (32, 10**6, "normal")]:
    print(f"  {name:<7} {lo:>3}-{hi if hi < 10**6 else '+':<4} px   "
          f"{np.mean((sides >= lo) & (sides < hi)):6.1%}")
print(f"  median {np.median(sides):.1f} px, p10 {np.percentile(sides, 10):.1f}, "
      f"p90 {np.percentile(sides, 90):.1f}")

probe_clips = sample_clips(train[:8], length=CLIP_LEN, stride=CLIP_STRIDE, size=SIZE,
                           frame_size=(width, height), jitter=32, seed=SEED)
native = sum(c.native for c in probe_clips)
print(f"\n{len(probe_clips)} clips from the first 8 sequences")
print(f"  {native / len(probe_clips):.1%} on native pixels -- the deployment's crop512")
print(f"  {1 - native / len(probe_clips):.1%} fall back to the resized full frame")

## Mask labels

Anti-UAV410 gives **boxes**; EdgeTAM predicts **masks**. A large SAM 2.1 teacher
runs once, offline, box-prompted per frame, and its masks become the training
target — the same relationship EdgeTAM already has to SAM 2, applied to one
domain.

Two things carry the quality:

- **Zoom.** Prompting the teacher on the full 640×512 frame asks it to segment a
  6-pixel object, and it will not. Prompting on a crop a few times the box size
  is the same model on a much easier problem.
- **Gates.** A mask is kept only if four independent checks agree: the teacher's
  own confidence, agreement with the human box, plausible area, and being one
  connected object. Frames that fail keep their `exist` supervision and fall
  back to a box-projection loss.

The acceptance rate below is a **measurement, not an assumption** — and *which*
gate rejects tells you what to fix.

In [ ]:
from src.training.labels import Gates, Sam2Teacher, label_sequence

GATES   = Gates(teacher_iou=0.7, box_iou=0.6, area=(0.15, 1.3), component=0.8)
teacher = Sam2Teacher(TEACHER_ID, device="cuda")
print(f"teacher {TEACHER_ID} on {torch.cuda.get_device_name(0)}")

In [ ]:
# --- One sequence first, before committing to all of them ---------------
import time
t0 = time.time()
probe = label_sequence(train[0], teacher, LABELS / "train", gates=GATES, zoom=ZOOM,
                       min_size=MIN_CROP, frame_size=(width, height),
                       stride=LABEL_STRIDE, batch_size=TEACHER_BATCH)
elapsed = time.time() - t0

print(probe)
print(f"\n{elapsed:.0f}s for {probe['attempted']} frames "
      f"({1000 * elapsed / max(probe['attempted'], 1):.0f} ms/frame)")
from src.training import frames_to_label
attempts = sum(len(frames_to_label(s, LABEL_STRIDE)) for s in train + val)
print(f"~{attempts * elapsed / max(probe['attempted'], 1) / 60:.0f} min for all "
      f"{len(train) + len(val)} sequences")

assert probe["acceptance_rate"] > 0.3, (
    "fewer than a third of frames produced a usable mask. Raise ZOOM or MIN_CROP, "
    "or loosen the gate named most often in probe['rejected'], before spending "
    "an hour on the rest.")

In [ ]:
# --- Look at them ------------------------------------------------------
import cv2
import matplotlib.pyplot as plt
from src.training import load_window, open_masks

masks = open_masks(LABELS / "train" / train[0].name / "pseudo_masks.npz")
picked = sorted(masks)[::max(len(masks) // 6, 1)][:6]

fig, axes = plt.subplots(1, len(picked), figsize=(3 * len(picked), 3.4))
for ax, idx in zip(np.atleast_1d(axes), picked):
    box = train[0].labels.boxes[idx]
    x0, y0 = max(int(box[0]) - 40, 0), max(int(box[1]) - 40, 0)
    w, h = int(box[2] - box[0]) + 80, int(box[3] - box[1]) + 80
    ax.imshow(load_window(train[0].frames[idx], (x0, y0), (w, h), 128))
    ax.imshow(cv2.resize(masks[idx][y0:y0 + h, x0:x0 + w].astype(np.uint8),
                         (128, 128), interpolation=cv2.INTER_NEAREST),
              alpha=0.45, cmap="autumn")
    ax.set_title(f"frame {idx}"); ax.axis("off")
plt.suptitle(f"{train[0].name}: teacher masks that passed all four gates")
plt.tight_layout(); plt.show()

In [ ]:
# --- The whole subset --------------------------------------------------
# Resumable: a sequence already labelled at this stride is skipped, so a
# dropped Colab connection costs the current sequence and nothing else.
import json
from src.training.labels import REPORT_FILE, summarise
from tqdm.auto import tqdm

def existing(split, sequence):
    path = LABELS / split / sequence.name / REPORT_FILE
    if not path.is_file():
        return None
    report = json.loads(path.read_text())
    ok = report.get("stride") == LABEL_STRIDE and report.get("frames") == len(sequence)
    return report if ok else None

reports, reused = {}, 0
for split, sequences in (("train", train), ("val", val)):
    rows = []
    for sequence in tqdm(sequences, desc=f"labelling {split}"):
        report = existing(split, sequence)
        reused += report is not None
        rows.append(report or label_sequence(
            sequence, teacher, LABELS / split, gates=GATES, zoom=ZOOM,
            min_size=MIN_CROP, frame_size=(width, height),
            stride=LABEL_STRIDE, batch_size=TEACHER_BATCH))
    reports[split] = rows
    print(f"\n### {split}\n{summarise(rows)}\n")
print(f"({reused} sequence(s) reused from an earlier run)")

In [ ]:
# --- Record what was built ---------------------------------------------
manifest = {
    "dataset": "Anti-UAV410", "data_root": str(DATA),
    "frame_size": [width, height], "model_input": SIZE,
    "clip": {"length": CLIP_LEN, "stride": CLIP_STRIDE},
    "teacher": TEACHER_ID, "label_stride": LABEL_STRIDE,
    "zoom": ZOOM, "min_crop": MIN_CROP,
    "gates": {"teacher_iou": GATES.teacher_iou, "box_iou": GATES.box_iou,
              "area": list(GATES.area), "component": GATES.component},
    "sequences": {k: [r["sequence"] for r in v] for k, v in reports.items()},
    "acceptance": {k: sum(r["accepted"] for r in v)
                      / max(sum(r.get("attempted", r["visible"]) for r in v), 1)
                   for k, v in reports.items()},
}
(WORK / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest["acceptance"], indent=2))

In [ ]:
# --- Give the GPU back --------------------------------------------------
# hiera-large is ~2 GB of weights plus its activations. Leaving it resident
# would come straight off the training batch size measured two cells below.
import gc
del teacher
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB allocated, "
      f"{torch.cuda.memory_reserved() / 2**30:.2f} GiB reserved")

## Önce kontrastı ölç

Ham parlaklık tek başına yeterli değildir: sıcak hedef de soğuk hedef de
izlenebilir. Ölçülen büyüklük, hedef kutusunun ortalama yoğunluğu ile kutunun
etrafındaki halka arasındaki mutlak farkın halka standart sapmasına oranıdır.
Düşük değer, hedefin yerel arka plan içinde kamufle olduğunu gösterir.

Bu ölçüm yalnız eğitim örnekleme ağırlığını ve rapor alt grubunu belirler;
test etiketleri eğitim kararına girmez.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def frame_local_contrast(sequence, index):
    image = cv2.imread(str(sequence.frames[index]), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return np.nan
    x0, y0, x1, y1 = sequence.labels.boxes[index]
    if not np.isfinite([x0, y0, x1, y1]).all():
        return np.nan
    h, w = image.shape
    bw, bh = max(x1 - x0, 2), max(y1 - y0, 2)
    pad = max(6, int(round(max(bw, bh))))
    ax0, ay0 = max(0, int(x0) - pad), max(0, int(y0) - pad)
    ax1, ay1 = min(w, int(np.ceil(x1)) + pad), min(h, int(np.ceil(y1)) + pad)
    tx0, ty0 = max(0, int(x0)), max(0, int(y0))
    tx1, ty1 = min(w, int(np.ceil(x1))), min(h, int(np.ceil(y1)))
    target = image[ty0:ty1, tx0:tx1].astype(np.float32)
    patch = image[ay0:ay1, ax0:ax1].astype(np.float32)
    if target.size < 4 or patch.size <= target.size:
        return np.nan
    ring = np.ones(patch.shape, dtype=bool)
    ring[ty0 - ay0:ty1 - ay0, tx0 - ax0:tx1 - ax0] = False
    background = patch[ring]
    if background.size < 8:
        return np.nan
    return float(abs(target.mean() - background.mean()) /
                 max(background.std(), 3.0))

def sequence_contrast(sequence, stride=CONTRAST_AUDIT_STRIDE):
    visible = sequence.labels.visible_indices()[::max(int(stride), 1)]
    values = [frame_local_contrast(sequence, int(i)) for i in visible]
    values = np.asarray([v for v in values if np.isfinite(v)], dtype=np.float32)
    return float(np.median(values)) if values.size else float("nan")

TRAIN_CONTRAST = {s.name: sequence_contrast(s) for s in train}
VAL_CONTRAST = {s.name: sequence_contrast(s) for s in val}
usable = np.asarray([v for v in TRAIN_CONTRAST.values() if np.isfinite(v)])
LOW_CONTRAST_CUT = float(np.quantile(usable, LOW_CONTRAST_QUANTILE))
LOW_TRAIN_NAMES = {name for name, value in TRAIN_CONTRAST.items()
                   if np.isfinite(value) and value <= LOW_CONTRAST_CUT}

print(f"train median local contrast: {np.median(usable):.3f}")
print(f"lowest {LOW_CONTRAST_QUANTILE:.0%} cut: {LOW_CONTRAST_CUT:.3f}")
print(f"oversampled sequences ({len(LOW_TRAIN_NAMES)}): {sorted(LOW_TRAIN_NAMES)}")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(usable, bins=20, color="#cc6b49")
ax.axvline(LOW_CONTRAST_CUT, color="black", ls="--", lw=1)
ax.set_xlabel("|target mean - ring mean| / ring std")
ax.set_ylabel("train sequences")
ax.set_title("Gerçek termal dizilerde yerel kontrast")
plt.tight_layout(); plt.show()

## Stage C: 22'nin checkpoint'inden temporal devam eğitimi

Bellek yolu çalışır ve modelin kendi önceki tahminleri sonraki kareyi
koşullandırır; teacher forcing yoktur. Bellek attention/encoder ağırlıkları
yine dondurulur: amaç bellek koordinat sistemini bozmak değil, görüntü
özelliklerini, maske/IoU başını ve nesne-güven skorunu termal kliplerde
kalibre etmektir. Kötü belleğin yazılıp yazılmaması ayrıca SAMURAI kolunda
ölçülür.

In [ ]:
from sam2.build_sam import build_sam2_video_predictor
from src.trackers._hydra_overrides import image_size_overrides

model = build_sam2_video_predictor(
    "configs/edgetam.yaml", str(BASE_STAGE_B), device="cuda",
    hydra_overrides_extra=image_size_overrides(SIZE),
)
model.eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M params | "
      f"start = {BASE_STAGE_B.name}")

In [ ]:
from src.training.finetune import Rates, apply_freeze, param_groups, summarise_freeze

print(summarise_freeze(apply_freeze(model, "head"), model))

# Assert the policy rather than trusting the print: a run that quietly trained
# the memory path would only show up as a bad checkpoint hours later.
for name, p in model.named_parameters():
    if name.startswith(("memory_attention", "memory_encoder", "spatial_perceiver")):
        assert not p.requires_grad, name

In [ ]:
# --- Video clips + real low-contrast oversampling ----------------------
from src.training import open_masks

def build(split, sequences, jitter):
    stores = {s.name: open_masks(LABELS / split / s.name / "pseudo_masks.npz")
              for s in sequences}
    clips = sample_clips(sequences, length=CLIP_LEN, stride=CLIP_STRIDE,
                         size=SIZE, frame_size=(width, height),
                         jitter=jitter, seed=SEED)
    return clips, stores

train_clips, train_stores = build("train", train, jitter=32)
val_clips, val_stores = build("val", val, jitter=0)
original_train_clips = len(train_clips)
hard = [clip for clip in train_clips if clip.sequence.name in LOW_TRAIN_NAMES]
train_clips = train_clips + hard * max(LOW_CONTRAST_REPEAT - 1, 0)

labelled = sum(len(store) for store in train_stores.values())
print(f"train {original_train_clips} -> {len(train_clips)} clips after "
      f"low-contrast x{LOW_CONTRAST_REPEAT} oversampling")
print(f"val {len(val_clips)} clips | {labelled} teacher-labelled train frames")

## Zamansal düşük-kontrast dönüşümü

Dönüşüm normalize edilmiş tensörü tekrar 0–1 yoğunluğa getirir. Global
kontrast/parlaklık klip boyunca yumuşak değişir; hedef kutusunun içi çevre
halkasının ortalamasına doğru çekilerek gerçek “hedef ile zemin aynı tona
geldi” örneği oluşturulur. Hafif sensör gürültüsü ve blur eklenebilir.

Geometriye dokunulmadığı için maskeler/kutular geçerli kalır. Validation ve
test asla augment edilmez.

In [ ]:
from dataclasses import dataclass, replace
import torch.nn.functional as F
from src.training.antiuav import MEAN, STD
from src.training.clip_loop import clip_losses
from src.training.losses import Weights
from src.training.schedule import CLIPS, Loop

TRACK_WEIGHTS = Weights(focal=20.0, dice=1.0, iou=2.0,
                        object_score=2.0, box_projection=1.0)

def uniform(shape, limits, device, generator):
    lo, hi = (float(v) for v in limits)
    return lo + (hi - lo) * torch.rand(
        shape, device=device, generator=generator)

def temporal_pair(batch, limits, generator):
    shape = (batch, 1, 1, 1, 1)
    return (uniform(shape, limits, "cuda", generator),
            uniform(shape, limits, "cuda", generator))

def low_contrast_augment(batch, generator):
    images = batch.images
    device = images.device
    b, t, _, h, w = images.shape
    mean = torch.as_tensor(MEAN, device=device).view(1, 1, 3, 1, 1)
    std = torch.as_tensor(STD, device=device).view(1, 1, 3, 1, 1)
    gray = (images * std + mean).clamp(0, 1).mean(2, keepdim=True)

    chosen = (torch.rand((b, 1, 1, 1, 1), device=device,
                         generator=generator) < AUGMENT_PROB).float()
    phase = torch.linspace(0, 1, t, device=device).view(1, t, 1, 1, 1)

    c0, c1 = temporal_pair(b, GLOBAL_CONTRAST, generator)
    contrast = c0 + (c1 - c0) * phase
    center = gray.mean((-2, -1), keepdim=True)
    augmented = center + contrast * (gray - center)

    b0, b1 = temporal_pair(b, BRIGHTNESS_SHIFT, generator)
    augmented = augmented + b0 + (b1 - b0) * phase

    # Hedefi çevresindeki termal tona yaklaştır. Kutunun yumuşatılmış
    # maskesi dikdörtgen kenar artefaktı oluşmasını engeller.
    boxes = batch.boxes
    xx = torch.arange(w, device=device).view(1, 1, 1, w)
    yy = torch.arange(h, device=device).view(1, 1, h, 1)
    x0, y0, x1, y1 = [boxes[..., i].unsqueeze(-1).unsqueeze(-1)
                      for i in range(4)]
    valid = batch.exist.bool().unsqueeze(-1).unsqueeze(-1)
    inside = valid & (xx >= x0) & (xx < x1) & (yy >= y0) & (yy < y1)
    bw = (x1 - x0).abs().clamp(min=4)
    bh = (y1 - y0).abs().clamp(min=4)
    pad = torch.maximum(bw, bh)
    outer = valid & (xx >= x0 - pad) & (xx < x1 + pad) & \
            (yy >= y0 - pad) & (yy < y1 + pad)
    ring = outer & ~inside
    ring5 = ring.unsqueeze(2)
    ring_mean = ((gray * ring5).sum((-2, -1), keepdim=True) /
                 ring5.sum((-2, -1), keepdim=True).clamp(min=1))

    tc0, tc1 = temporal_pair(b, TARGET_CONTRAST, generator)
    target_factor = tc0 + (tc1 - tc0) * phase
    suppressed = ring_mean + target_factor * (augmented - ring_mean)
    soft = F.avg_pool2d(inside.float().reshape(b * t, 1, h, w),
                        kernel_size=11, stride=1, padding=5)
    soft = soft.reshape(b, t, 1, h, w).clamp(0, 1) * chosen
    augmented = augmented * (1 - soft) + suppressed * soft

    sigma = uniform((b, 1, 1, 1, 1), SENSOR_NOISE, device, generator)
    noise = torch.randn(gray.shape, device=device, generator=generator)
    augmented = augmented + chosen * sigma * noise

    blurred = F.avg_pool2d(augmented.reshape(b * t, 1, h, w),
                           kernel_size=3, stride=1, padding=1)
    blurred = blurred.reshape(b, t, 1, h, w)
    blur = ((torch.rand((b, 1, 1, 1, 1), device=device,
                        generator=generator) < BLUR_PROB).float() * chosen)
    augmented = augmented * (1 - blur) + blurred * blur

    gray = (gray * (1 - chosen) + augmented * chosen).clamp(0, 1)
    rgb = gray.expand(-1, -1, 3, -1, -1)
    return replace(batch, images=(rgb - mean) / std)

@dataclass(frozen=True)
class ContrastSplit:
    clips: list
    stores: dict
    augment: bool

def contrast_stream(split, batch, seed, limit, device="cuda",
                    workers=8, depth=2):
    stream = CLIPS.stream(split, batch, seed, limit, device, workers, depth)
    generator = torch.Generator(device=device)
    generator.manual_seed(SEED if seed is None else int(seed))
    for item in stream:
        yield low_contrast_augment(item, generator) if split.augment else item

def tracking_loss(model, batch):
    return clip_losses(model, batch, weights=TRACK_WEIGHTS)

CONTRAST_LOOP = Loop(stream=contrast_stream, loss=tracking_loss,
                     val_loss=tracking_loss)
TRAIN_SPLIT = ContrastSplit(train_clips, train_stores, True)
VAL_SPLIT = ContrastSplit(val_clips, val_stores, False)

# Görsel smoke test: solda gerçek, sağda aynı klibin augment edilmiş hali.
preview = next(CLIPS.stream(VAL_SPLIT, 3, SEED, 1, "cuda", 2, 1))
generator = torch.Generator(device="cuda").manual_seed(1234)
changed = low_contrast_augment(preview, generator)
mean = torch.as_tensor(MEAN, device="cuda").view(1, 1, 3, 1, 1)
std = torch.as_tensor(STD, device="cuda").view(1, 1, 3, 1, 1)
before = (preview.images * std + mean).clamp(0, 1).cpu()
after = (changed.images * std + mean).clamp(0, 1).cpu()
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for k in range(3):
    axes[0, k].imshow(before[k, 0].permute(1, 2, 0)); axes[0, k].axis("off")
    axes[1, k].imshow(after[k, 0].permute(1, 2, 0)); axes[1, k].axis("off")
axes[0, 0].set_ylabel("real"); axes[1, 0].set_ylabel("augmented")
plt.tight_layout(); plt.show()
del preview, changed, before, after
torch.cuda.empty_cache()

## First: overfit one clip

Before spending GPU hours, prove the loop can learn *anything*. One clip, no
augmentation, frame 0 included in the loss — this should collapse towards zero
within a couple of hundred steps.

If it does not, the problem is in the plumbing (prompt coordinates, mask
alignment, the memory bookkeeping), and no amount of real training will fix it.
This is the cheapest possible place to find that out.

In [ ]:
import copy
from src.training.clip_loop import clip_losses, collate

# The clip with the most of *its own* frames labelled -- a sequence-level count
# would happily pick a clip whose eight frames all fell in the labelling stride.
def labelled_frames(clip):
    store = train_stores[clip.sequence.name]
    return sum(int(i) in store for i in clip.indices)

probe_clip = max(train_clips[:4000], key=labelled_frames)
probe = collate([probe_clip], [train_stores[probe_clip.sequence.name]], "cpu").to("cuda")
print(f"{probe_clip.sequence.name} frames {probe_clip.indices}, "
      f"{labelled_frames(probe_clip)} of {CLIP_LEN} with a teacher mask")

snapshot = copy.deepcopy(model.state_dict())
opt = torch.optim.AdamW(param_groups(model, Rates(head=3e-4)))
trainable = [p for g in opt.param_groups for p in g["params"]]
history = []
for step in range(120):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss, terms = clip_losses(model, probe, skip_first=False)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
    opt.step()
    history.append(float(loss))
    if step % 20 == 0:
        print(f"step {step:>4}  loss {float(loss):7.4f}   " +
              "  ".join(f"{k} {v:.3f}" for k, v in terms.items()))

drop = history[0] / max(min(history[-10:]), 1e-6)
print(f"\nloss fell {drop:.1f}x  ({history[0]:.3f} -> {min(history[-10:]):.3f})")
assert drop > 3.0, ("the loop cannot even overfit one clip. Check prompt "
                    "coordinates, mask alignment and the memory bookkeeping "
                    "before training on anything larger.")
model.load_state_dict(snapshot)          # throw the overfit away
del snapshot, opt, probe
gc.collect(); torch.cuda.empty_cache()

## How many clips at once

**The batch dimension is clips, not frames.** SAM 2 batches *objects*, and each
row of that batch carries its own memory, its own object pointer and its own
object score — the attention never mixes rows. So N independent clips ride the
same machinery N tracked objects would, with no change to the model.

The largest N is a property of the card, not of the recipe, and it is measured
rather than guessed: a real forward *and backward* at each candidate size —
the backward is where the activation graph is actually held — and the largest
one that leaves 15 % of the card free for fragmentation, the EMA copy and the
validation pass.

The other half of using a big batch is **keeping it fed**. One batch of 16 clips
is 128 JPEGs to decode, crop and normalise: ~0.4 s of OpenCV against ~0.15 s of
compute, so on the training thread the GPU would idle two thirds of every step
and a bigger card would buy nothing. `prefetch` moves that onto worker threads —
`cv2.imread` releases the GIL, so it is real parallelism — and hands the loop
batches that are already assembled.

`LOADER_WORKERS` and `PREFETCH_DEPTH` pull in different directions and are
separate for that reason. Workers is how many clips are read at once and wants
to be around the core count; depth is how many *batches* sit in host RAM ahead
of the GPU and wants to stay at two. At `BATCH = 64` one batch is 1.6 GB, so a
queue length tied to the thread count would be tens of gigabytes of decoded
JPEG waiting on a card that is content with two.

In [ ]:
from src.training.loader import auto_batch_size, batch_clips, prefetch

# Measure under the *encoder* stage's freeze, which is the expensive one: with
# only the head trainable the trunk's activations are never kept for a
# backward, so a batch sized there would OOM the moment the encoder unfreezes.
apply_freeze(model, "encoder")
BATCH = auto_batch_size(model, train_clips, train_stores, maximum=BATCH_CEILING)
apply_freeze(model, "head")
ACCUM = 1                       # the measured batch is already large; no need

print(f"\ntraining on {BATCH} clips x {CLIP_LEN} frames = "
      f"{BATCH * CLIP_LEN} frames per step")

In [ ]:
# --- Loader + augmentation throughput ----------------------------------
import time

stream = contrast_stream(TRAIN_SPLIT, BATCH, SEED, 6, "cuda",
                         LOADER_WORKERS, PREFETCH_DEPTH)
held = next(stream)
t0 = time.time()
for batch in stream:
    del batch
load = (time.time() - t0) / 5

t0 = time.time()
for _ in range(4):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        tracking_loss(model, held)[0].backward()
    model.zero_grad(set_to_none=True)
torch.cuda.synchronize()
step = (time.time() - t0) / 4

print(f"load+augment {load * 1000:6.0f} ms/batch")
print(f"compute      {step * 1000:6.0f} ms/batch")
print("GPU-bound" if step > load else "input-bound: raise LOADER_WORKERS")
del held; gc.collect(); torch.cuda.empty_cache()

## Eğitim

İlk aşama başlık/IoU/object-score kalibrasyonunu yapar. İkinci aşama
encoder'ı daha düşük hızla açar. `iou` ve `object_score` ağırlıkları 2'dir;
çünkü bunlar yalnız rapor sayıları değil, aday maske seçimi ve bellek kapısının
güven sinyalleridir. Validation gerçek görüntülerde, augmentasyonsuz yapılır.

In [ ]:
from src.training.finetune import save_checkpoint
from src.training.schedule import Schedule, run_stages

schedule = Schedule(
    stages=(("head", 2, Rates(head=1e-4)),
            ("encoder", 3, Rates(head=5e-5, neck=5e-5, trunk=1e-5))),
    batch=BATCH, accum=ACCUM, steps_per_epoch=STEPS_PER_EPOCH,
    val_batches=VAL_BATCHES, workers=LOADER_WORKERS,
    depth=PREFETCH_DEPTH, seed=SEED, patience=2,
    meta={
        "method": "temporal_contrast_finetune", "image_size": SIZE,
        "dataset": "Anti-UAV410", "base": str(BASE_STAGE_B),
        "teacher": TEACHER_ID, "label_stride": LABEL_STRIDE,
        "augment": {
            "prob": AUGMENT_PROB,
            "global_contrast": GLOBAL_CONTRAST,
            "target_contrast": TARGET_CONTRAST,
            "brightness": BRIGHTNESS_SHIFT,
            "noise": SENSOR_NOISE,
            "blur_prob": BLUR_PROB,
        },
        "loss_weights": TRACK_WEIGHTS.__dict__,
    })

result = run_stages(
    model, TRAIN_SPLIT, VAL_SPLIT, schedule,
    freeze=apply_freeze,
    save=lambda m, meta: save_checkpoint(m, CHECKPOINT, meta),
    progress=lambda stream, total, desc: tqdm(stream, total=total, desc=desc),
    loop=CONTRAST_LOOP,
)
assert CHECKPOINT.is_file(), "training produced no checkpoint"
print(f"best val clip loss {result['best_val_loss']:.4f} -> {CHECKPOINT}")

In [ ]:
# --- Free the training graph before the evaluation runs -----------------
# tools/eval_antiuav.py starts its own tracker in a subprocess; this model's
# weights are already on disk and its optimiser state is worth nothing now.
del model, train_clips, val_clips
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.memory_reserved() / 2**30:.2f} GiB reserved")

## Takip değerlendirmesi: üç kol

Tek-kare IoU yerine deployment yolunun tamamı çalıştırılır:

- **stage B:** `22` checkpoint'i, stok FIFO bellek;
- **temporal:** yeni checkpoint, stok FIFO bellek;
- **temporal + samurai:** aynı yeni checkpoint, hareket-duyarlı aday seçimi
  ve yalnız güvenilir kareleri kabul eden bellek kapısı.

Böylece ağırlık eğitiminin ve bellek politikasının katkıları birbirine
karıştırılmaz. Rapor tüm validation yanında gerçek yerel kontrastı en düşük
dörtte birlik grubu ayrıca gösterir.

In [ ]:
import subprocess, yaml

def tracker_config(checkpoint, samurai=None):
    cfg = {
        "model_cfg": "configs/edgetam.yaml",
        "checkpoint": str(checkpoint),
        "image_size": SIZE, "device": "cuda", "precision": "bfloat16",
        "mask_threshold": 0.0,
        "offload_video_to_cpu": False, "offload_state_to_cpu": False,
    }
    if samurai is not None:
        cfg["samurai"] = samurai
    return cfg

EVAL_CONFIGS = {
    "stage_b": WORK / "eval_stage_b.yaml",
    "temporal": WORK / "eval_temporal.yaml",
    "temporal+samurai": WORK / "eval_temporal_samurai.yaml",
}
EVAL_CONFIGS["stage_b"].write_text(yaml.safe_dump(
    tracker_config(BASE_STAGE_B), sort_keys=False))
EVAL_CONFIGS["temporal"].write_text(yaml.safe_dump(
    tracker_config(CHECKPOINT), sort_keys=False))
EVAL_CONFIGS["temporal+samurai"].write_text(yaml.safe_dump(
    tracker_config(CHECKPOINT, SAMURAI), sort_keys=False))

def run_eval(label, split, limit):
    output = WORK / f"eval_{split}_{label.replace('+', '_')}.json"
    command = [sys.executable, "tools/eval_antiuav.py",
               "--data", str(DATA), "--split", split,
               "--tracker", "edgetam", "--config", str(EVAL_CONFIGS[label]),
               "--mode", "crop", "--size", str(SIZE),
               "--json", str(output)]
    if limit is not None:
        command += ["--limit", str(limit)]
    subprocess.run(command, check=True)
    return output

VAL_FILES = {label: run_eval(label, "val", VAL_SEQUENCES)
             for label in EVAL_CONFIGS}

In [ ]:
def load_rows(path):
    return json.loads(Path(path).read_text())["sequences"]

def weighted(rows, key):
    frames = sum(row["frames"] for row in rows)
    return sum(row[key] * row["frames"] for row in rows) / max(frames, 1)

def summary(rows):
    return {
        "state_accuracy": weighted(rows, "state_accuracy"),
        "success_auc": weighted(rows, "success_auc"),
        "lost_frames": sum(sum(row["dropout_lengths"]) for row in rows),
        "episodes": sum(len(row["dropout_lengths"]) for row in rows),
        "longest": max((max(row["dropout_lengths"], default=0)
                        for row in rows), default=0),
    }

def print_group(title, rows_by_label, contrast):
    names = [name for name, value in contrast.items() if np.isfinite(value)]
    names.sort(key=lambda name: contrast[name])
    low = set(names[:max(1, int(np.ceil(len(names) * 0.25)))])
    print(f"\n{title}")
    print(f"{'arm':<20}{'group':<10}{'state acc':>11}{'AUC':>9}"
          f"{'lost':>9}{'episodes':>10}{'longest':>9}")
    for label, rows in rows_by_label.items():
        for group, selected in (("all", rows),
                                ("low-25%", [r for r in rows if r['name'] in low])):
            s = summary(selected)
            print(f"{label:<20}{group:<10}{s['state_accuracy']:>11.4f}"
                  f"{s['success_auc']:>9.4f}{s['lost_frames']:>9}"
                  f"{s['episodes']:>10}{s['longest']:>9}")
    print("low-contrast sequences:", sorted(low))

VAL_ROWS = {label: load_rows(path) for label, path in VAL_FILES.items()}
print_group("validation", VAL_ROWS, VAL_CONTRAST)

# Teste yalnız val'de seçilen temporal bellek politikası gider.
SELECTED_LABEL = max(("temporal", "temporal+samurai"),
                     key=lambda label: summary(VAL_ROWS[label])["state_accuracy"])
print("\nselected on val:", SELECTED_LABEL)

### Test: bir kez

Test, augmentation ayarı veya SAMURAI seçimi yapmak için kullanılmaz. Seçim
validation state accuracy ile yukarıda yapılmıştır. Burada yalnız stage-B
başlangıcı ve seçilmiş final kolu karşılaştırılır.

In [ ]:
assert "test" in splits, "test split indirilmedi"
test_sequences = list_sequences(DATA, "test")[:TEST_SEQUENCES]
TEST_CONTRAST = {s.name: sequence_contrast(s) for s in test_sequences}
TEST_FILES = {
    "stage_b": run_eval("stage_b", "test", TEST_SEQUENCES),
    SELECTED_LABEL: run_eval(SELECTED_LABEL, "test", TEST_SEQUENCES),
}
TEST_ROWS = {label: load_rows(path) for label, path in TEST_FILES.items()}
print_group("test (held out)", TEST_ROWS, TEST_CONTRAST)

In [ ]:
# --- Kalıcı çıktılar ----------------------------------------------------
import shutil

deploy = tracker_config(
    "checkpoints/edgetam_thermal_contrast_tracking_512.pt",
    SAMURAI if SELECTED_LABEL == "temporal+samurai" else None)
DEPLOY_CONFIG = WORK / "edgetam_thermal_contrast_tracking_512.yaml"
DEPLOY_CONFIG.write_text(yaml.safe_dump(deploy, sort_keys=False))

summary_file = WORK / "contrast_tracking_log.json"
summary_file.write_text(json.dumps({
    "base_stage_b": str(BASE_STAGE_B), "checkpoint": str(CHECKPOINT),
    "selected_on_val": SELECTED_LABEL,
    "low_contrast_cut": LOW_CONTRAST_CUT,
    "low_train_sequences": sorted(LOW_TRAIN_NAMES),
    "train_sequence_contrast": TRAIN_CONTRAST,
    "val_sequence_contrast": VAL_CONTRAST,
    "test_sequence_contrast": TEST_CONTRAST,
    "training": result,
    "validation": {k: summary(v) for k, v in VAL_ROWS.items()},
    "test": {k: summary(v) for k, v in TEST_ROWS.items()},
}, indent=2) + "\n")

if MIRROR is not None:
    shutil.copy(CHECKPOINT, MIRROR / CHECKPOINT.name)
    shutil.copy(DEPLOY_CONFIG, MIRROR / DEPLOY_CONFIG.name)
    shutil.copy(summary_file, MIRROR / summary_file.name)
    shutil.copy(WORK / "manifest.json", MIRROR / "manifest.json")
    for path in sorted(WORK.glob("eval_*.json")):
        shutil.copy(path, MIRROR / path.name)
    for split in ("train", "val"):
        shutil.copytree(LABELS / split, MIRROR / "labels" / split,
                        dirs_exist_ok=True)
    print("saved to", MIRROR)
else:
    print("download before runtime ends:", CHECKPOINT, DEPLOY_CONFIG,
          summary_file)

## Sonucu nasıl okuyacaksınız?

En önemli satır `low-25%` grubudur.

- `temporal`, stage B'den iyi ve kayıp episode'ları daha kısa ise düşük
  kontrast eğitimi işe yaramıştır.
- `temporal+samurai` ayrıca iyiyse sorun yalnız temsil değil, kötü karenin
  belleğe yazılmasıdır; SAMURAI YAML'ı ile deploy edin.
- Genel skor artıp `low-25%` artmıyorsa kontrast dönüşümü veri dağılımını
  yakalamıyordur. Önce görsel smoke test'i ve gerçek kamera histogramlarını
  karşılaştırın; `GLOBAL_CONTRAST` değerini körlemesine daha da düşürmeyin.
- AUC iyi ama `lost/longest` yüksek kalıyorsa maske şekli düzelmiş, kimlik
  sürekliliği düzelmemiştir. Bu durumda daha fazla **video** ve benzer hedefli
  dizi gerekir; statik havuz eklemek tek başına doğru ekseni büyütmez.

Final checkpoint mimariyi değiştirmez. TensorRT export edilecekse yeni
checkpoint'ten yeniden export/kalibrasyon yapın; eski engine'lerle yeni
checkpoint'i karıştırmayın.